In [0]:
%sql
DROP TABLE ecommerce.gold.pipeline_logs;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.pipeline_logs (
    log_timestamp TIMESTAMP,
    layer STRING,
    table_name STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    record_count LONG,
    message STRING
)
USING DELTA

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS ecommerce.gold.export_logs;

In [0]:
from pyspark.sql.functions import current_timestamp
from datetime import datetime

def log_pipeline(layer, table_name, status, start_time, end_time, record_count, message):

    log_data = [(layer, table_name, status, start_time, end_time, record_count, message)]

    columns = [
        "layer",
        "table_name",
        "status",
        "start_time",
        "end_time",
        "record_count",
        "message"
    ]

    log_df = spark.createDataFrame(log_data, columns) \
        .withColumn("log_timestamp", current_timestamp())

    log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.pipeline_logs")

    export_log_path = f"/Volumes/ecommerce/gold/export_logs/{layer}_pipeline.log"

    log_string = f"{datetime.now()} | {layer} | {table_name} | {status} | {record_count} | {message}\n"

    try:
        existing_log = dbutils.fs.head(export_log_path)
        new_log = existing_log + log_string
    except:
        new_log = log_string

    dbutils.fs.put(export_log_path, new_log, True)

###===============================
##dim_Customer Table
###===============================

In [0]:
%sql
DROP TABLE ecommerce.gold.dim_customer;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.dim_customer (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- Surrogate Key (PK)
    customer_id STRING NOT NULL, -- Natural Key
    customer_unique_id STRING NOT NULL,
    customer_zip_code_prefix INT,
    customer_city STRING,
    customer_state STRING
)
USING DELTA

In [0]:
%sql
ALTER TABLE ecommerce.gold.dim_customer
ADD CONSTRAINT pk_dim_customers
PRIMARY KEY (customer_sk);

In [0]:
from pyspark.sql.functions import col
from datetime import datetime

start_time = datetime.now()

try:

    customers = spark.table("ecommerce.silver.customers")

    dim_customer = (
        customers
                    
         # Select final columns
        .select(
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        )
    )

    record_count = dim_customer.count()

    dim_customer.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.dim_customer")

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_customer",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "dim_customer loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_customer",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###===============================
##dim_Seller Table
###===============================

In [0]:
%sql
DROP TABLE ecommerce.gold.dim_seller;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.dim_seller (
    seller_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- PK
    seller_id STRING NOT NULL, -- Natural Key
    seller_zip_code_prefix INT,
    seller_city STRING,
    seller_state STRING
)
USING DELTA

In [0]:
from pyspark.sql.functions import col
from datetime import datetime

start_time = datetime.now()

try:

    sellers = spark.table("ecommerce.silver.sellers")

    dim_seller = (
        sellers
                    
        # Select final columns
        .select(
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        )
    )

    record_count = dim_seller.count()

    dim_seller.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.dim_seller")

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_seller",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "dim_seller loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_seller",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###===============================
##dim_Product Table
###===============================

In [0]:
%sql
DROP TABLE ecommerce.gold.dim_product;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.dim_product (
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- PK
    product_id STRING NOT NULL,
    product_category_name STRING,
    product_name_lenght INT,
    product_description_lenght INT,
    product_photos_qty INT,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE
)
USING DELTA

In [0]:
from pyspark.sql.functions import col
from datetime import datetime

start_time = datetime.now()

try:

    products = spark.table("ecommerce.silver.product")

    dim_product = (
        products
        .withColumn("product_weight_g", col("product_weight_g").cast("double"))
        .withColumn("product_length_cm", col("product_length_cm").cast("double"))
        .withColumn("product_height_cm", col("product_height_cm").cast("double"))
        .withColumn("product_width_cm", col("product_width_cm").cast("double"))
    
        .select(
            "product_id",
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        )
    )

    record_count = dim_product.count()

    dim_product.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.dim_product")

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_product",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "dim_product loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_product",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###===============================
##dim_Category Table
###===============================

In [0]:
%sql
DROP TABLE ecommerce.gold.dim_category;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.dim_category (
    category_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- PK
    product_category_name STRING NOT NULL,
    product_category_name_english STRING
)
USING DELTA

In [0]:
from pyspark.sql.functions import col, current_timestamp
from datetime import datetime

start_time = datetime.now()

try:

    category = spark.table("ecommerce.silver.category")

    dim_category = (
        category
       
        .select(
            "product_category_name",
            "product_category_name_english"
        )
    )

    record_count = dim_category.count()

    dim_category.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.dim_category")

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_category",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "dim_category loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_category",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###===============================
##dim_Geo Table
###===============================

In [0]:
%sql
DROP TABLE ecommerce.gold.dim_geolocation;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS ecommerce.gold.dim_geolocation (
    geolocation_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- PK
    geolocation_zip_code_prefix INT NOT NULL,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city STRING,
    geolocation_state STRING
)
USING DELTA

In [0]:
from pyspark.sql.functions import col, current_timestamp
from datetime import datetime

start_time = datetime.now()

try:

    geolocation = spark.table("ecommerce.silver.geolocation")

    dim_geolocation = (
        geolocation

        .select(
           "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "geolocation_city",
            "geolocation_state"
        )
    )

    record_count = dim_geolocation.count()

    dim_geolocation.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.dim_geolocation")

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_geolocation",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "dim_geolocation loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_geolocation",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###===============================
##dim_payment_type Table
###===============================

In [0]:
%sql
DROP TABLE ecommerce.gold.dim_payment_type;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.dim_payment_type (
    payment_type_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- PK
    payment_type STRING NOT NULL
)
USING DELTA

In [0]:
from pyspark.sql.functions import col, current_timestamp
from datetime import datetime

start_time = datetime.now()

try:

    payment_type = spark.table("ecommerce.silver.payments")

    dim_payment_type = (
        payment_type
       
        .select(
           "payment_type"
        )
    )

    record_count = dim_payment_type.count()

    dim_payment_type.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.dim_payment_type")

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_payment_type",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "dim_payment_type loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_payment_type",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###===============================
##facts_payments Table
###===============================

In [0]:
%sql
DROP TABLE ecommerce.gold.fact_payments;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.fact_payments (
    payment_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- PK
    
    order_id STRING NOT NULL,
    payment_sequential INT NOT NULL,

    customer_sk BIGINT NOT NULL,
    payment_type_sk BIGINT NOT NULL,

    payment_installments INT,
    payment_value DOUBLE
)
USING DELTA

In [0]:

from pyspark.sql.functions import col, broadcast

spark.conf.set("spark.sql.shuffle.partitions", "50")
start_time = datetime.now()

try: 
    payments = spark.table("ecommerce.silver.payments") \
        .select("order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value").alias("p")
        # .repartition("order_id") 

    orders = spark.table("ecommerce.silver.orders") \
        .select("order_id", "customer_id").alias("o")
        # .repartition("order_id") \

    dim_payment_type = spark.table("ecommerce.gold.dim_payment_type").alias("dpt")
    dim_customer = spark.table("ecommerce.gold.dim_customer").alias("dc")

    fact_payments = (
        payments
        .join(broadcast(orders), col("p.order_id") == col("o.order_id"), "inner")
        .join(broadcast(dim_customer), col("o.customer_id") == col("dc.customer_id"), "inner")
        .join(broadcast(dim_payment_type), col("p.payment_type") == col("dpt.payment_type"), "inner")
        .select(
            col("p.order_id"),
            col("p.payment_sequential"),
            col("dc.customer_sk"),
            col("dpt.payment_type_sk"),
            col("p.payment_installments"),
            col("p.payment_value")
        )
    )
    recod_count = 0
    # fact_payments.write.mode("append").saveAsTable("ecommerce.gold.fact_payments")

    end_time = datetime.now()

    log_pipeline(
            "gold",
            "fact_payments",
            "SUCCESS",
            start_time,
            end_time,
            record_count,
            "fact_payments loaded successfully"
        )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "fact_payments",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

###===============================
##dim_Date Table
###===============================

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.dim_date (
    date_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- PK
    date DATE NOT NULL,
    year INT,
    month INT,
    day INT,
    quarter INT,
    week INT
)
USING DELTA

In [0]:
orders = spark.table("ecommerce.silver.orders")
order_items = spark.table("ecommerce.silver.order_items")
reviews = spark.table("ecommerce.silver.reviews")

In [0]:
from pyspark.sql.functions import to_date

orders = orders.select(
    to_date("order_purchase_timestamp").alias("order_purchase_timestamp"),
    to_date("order_approved_at").alias("order_approved_at"),
    to_date("order_delivered_carrier_date").alias("order_delivered_carrier_date"),
    to_date("order_delivered_customer_date").alias("order_delivered_customer_date"),
    to_date("order_estimated_delivery_date").alias("order_estimated_delivery_date")
)

order_items = order_items.select(
    to_date("shipping_limit_date").alias("shipping_limit_date")
)

reviews = reviews.select(
    to_date("review_creation_date").alias("review_creation_date"),
    to_date("review_answer_timestamp").alias("review_answer_timestamp")
)

In [0]:
from pyspark.sql.functions import min, max, least, greatest

# Aggregate min/max per table
orders_min_max = orders.select(
    least(
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ).alias("min_date"),
    greatest(
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ).alias("max_date")
)

order_items_min_max = order_items.select(
    min("shipping_limit_date").alias("min_date"),
    max("shipping_limit_date").alias("max_date")
)

reviews_min_max = reviews.select(
    least("review_creation_date", "review_answer_timestamp").alias("min_date"),
    greatest("review_creation_date", "review_answer_timestamp").alias("max_date")
)

# Union all and get global min/max
all_dates = (
    orders_min_max
    .union(order_items_min_max)
    .union(reviews_min_max)
)

final_dates = all_dates.agg(
    min("min_date").alias("start_date"),
    max("max_date").alias("end_date")
).collect()[0]

start_date = final_dates["start_date"]
end_date = final_dates["end_date"]

In [0]:
from pyspark.sql.functions import sequence, explode, col, year, month, dayofmonth, quarter, weekofyear
from datetime import datetime

start_time = datetime.now()
try:

    date_df = spark.createDataFrame([(start_date, end_date)], ["start", "end"])

    calendar = (
        date_df
        .select(explode(sequence(col("start"), col("end"))).alias("date"))
    )
    dim_date = (
        calendar
        .withColumn("year", year("date"))
        .withColumn("month", month("date"))
        .withColumn("day", dayofmonth("date"))
        .withColumn("quarter", quarter("date"))
        .withColumn("week", weekofyear("date"))

        .select(
           "date",
           "year",
           "month",
           "day",
           "quarter",
           "week"
        )
    )
    record_count = dim_date.count()

    dim_date.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.dim_date")

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_date",
        "SUCCESS",
        start_time,
        end_time,
        record_count,
        "dim_date loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "dim_date",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

In [0]:
%sql
SELECT * FROM ecommerce.gold.dim_date;


In [0]:
%sql
INSERT INTO ecommerce.gold.dim_geolocation (geolocation_sk, geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state)
VALUES (-1, -1,-0.0,-0.0,'UNKNOWN','UNKNOWN');

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.fact_sales (
    sales_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- PK
    
    -- Natural Keys (optional but useful)
    order_id STRING NOT NULL,
    order_item_id INT NOT NULL,

    -- Foreign Keys (Surrogate Keys)
    customer_sk BIGINT NOT NULL,
    product_sk BIGINT NOT NULL,
    seller_sk BIGINT NOT NULL,
    date_sk BIGINT NOT NULL,
    geolocation_sk BIGINT NOT NULL,
    category_sk  BIGINT NOT NULL,

    --Dates
    shipping_limit_date DATE,
    order_purchase_timestamp DATE,
    order_approved_at DATE,
    order_delivered_carrier_date DATE,
    order_delivered_customer_date DATE,
    order_estimated_delivery_date DATE,
    
    order_status STRING,
    -- Measures
    price DOUBLE,
    freight_value DOUBLE,
    review_score INT,
    
    -- Derived Metrics
    total_value DOUBLE,
    delivery_days INT,
    delivery_delay_flag BOOLEAN

    
)
USING DELTA

In [0]:
from pyspark.sql.functions import col, broadcast, to_date, datediff, when, coalesce, lit, avg
from datetime import datetime

spark.conf.set("spark.sql.shuffle.partitions", "50")

start_time = datetime.now()

try:

    orders = spark.table("ecommerce.silver.orders").alias("o")

    order_items = spark.table("ecommerce.silver.order_items") \
        .select(
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "shipping_limit_date",
            "price",
            "freight_value"
        ).alias("oi")

    reviews = spark.table("ecommerce.silver.reviews") \
        .groupBy("order_id") \
        .agg(avg("review_score").cast("int").alias("review_score")) \
        .alias("r")

    dim_customer = spark.table("ecommerce.gold.dim_customer").alias("dc")
    dim_product = spark.table("ecommerce.gold.dim_product").alias("dp")
    dim_seller = spark.table("ecommerce.gold.dim_seller").alias("ds")
    dim_date = spark.table("ecommerce.gold.dim_date").alias("dd")
    dim_geo = spark.table("ecommerce.gold.dim_geolocation").alias("dg")
    dim_category = spark.table("ecommerce.gold.dim_category").alias("dcat")

    fact_sales = (
        order_items

        .join(orders, col("oi.order_id") == col("o.order_id"), "inner")

        .join(broadcast(reviews), col("o.order_id") == col("r.order_id"), "left")

        .join(broadcast(dim_customer), col("o.customer_id") == col("dc.customer_id"), "inner")
        .join(broadcast(dim_product), col("oi.product_id") == col("dp.product_id"), "inner")
        .join(broadcast(dim_seller), col("oi.seller_id") == col("ds.seller_id"), "inner")

        .join(
            broadcast(dim_date),
            to_date(col("o.order_purchase_timestamp")) == col("dd.date"),
            "inner"
        )

        .join(
            broadcast(dim_geo),
            col("dc.customer_zip_code_prefix") == col("dg.geolocation_zip_code_prefix"),
            "left"
        )

        .join(
            broadcast(dim_category),
            col("dp.product_category_name") == col("dcat.product_category_name"),
            "inner"
        )

        .select(
            col("oi.order_id"),
            col("oi.order_item_id"),

            col("dc.customer_sk"),
            col("dp.product_sk"),
            col("ds.seller_sk"),
            col("dd.date_sk"),

            coalesce(col("dg.geolocation_sk"), lit(-1)).alias("geolocation_sk"),
            coalesce(col("dcat.category_sk"), lit(-1)).alias("category_sk"),

            # Dates
            to_date("oi.shipping_limit_date").alias("shipping_limit_date"),
            to_date("o.order_purchase_timestamp").alias("order_purchase_timestamp"),
            to_date("o.order_approved_at").alias("order_approved_at"),
            to_date("o.order_delivered_carrier_date").alias("order_delivered_carrier_date"),
            to_date("o.order_delivered_customer_date").alias("order_delivered_customer_date"),
            to_date("o.order_estimated_delivery_date").alias("order_estimated_delivery_date"),

            col("o.order_status"),

            # Measures
            col("oi.price"),
            col("oi.freight_value"),
            col("r.review_score"),

            # Derived
            (col("oi.price") + col("oi.freight_value")).alias("total_value"),

            datediff(
                col("o.order_delivered_customer_date"),
                col("o.order_purchase_timestamp")
            ).alias("delivery_days"),

            when(
                col("o.order_delivered_customer_date") > col("o.order_estimated_delivery_date"),
                True
            ).otherwise(False).alias("delivery_delay_flag")
        )
    )

    fact_sales.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.fact_sales")

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "fact_sales",
        "SUCCESS",
        start_time,
        end_time,
        0,
        "fact_sales loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "fact_sales",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.gold.fact_reviews (
    review_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- PK
    
    date_sk BIGINT NOT NULL,
    customer_sk BIGINT NOT NULL,
    product_sk BIGINT NOT NULL,

    review_id STRING NOT NULL,
    order_id STRING NOT NULL,

    review_creation_date DATE,
    review_answer_timestamp DATE,

    review_score INT,
    review_length INT,

    review_comment_title STRING,
    review_comment_message STRING
)
USING DELTA

In [0]:
from pyspark.sql.functions import col, broadcast, to_date, length, coalesce, lit
from datetime import datetime

spark.conf.set("spark.sql.shuffle.partitions", "50")

start_time = datetime.now()

try:

    reviews = spark.table("ecommerce.silver.reviews").alias("r")

    orders = spark.table("ecommerce.silver.orders") \
        .select("order_id", "customer_id") \
        .alias("o")

    order_items = spark.table("ecommerce.silver.order_items") \
        .select("order_id", "product_id") \
        .distinct() \
        .alias("oi")

    dim_customer = spark.table("ecommerce.gold.dim_customer").alias("dc")
    dim_date = spark.table("ecommerce.gold.dim_date").alias("dd")
    dim_product = spark.table("ecommerce.gold.dim_product").alias("dp")

    fact_reviews = (
        reviews

        .join(orders, col("r.order_id") == col("o.order_id"), "inner")

        .join(broadcast(dim_customer), col("o.customer_id") == col("dc.customer_id"), "inner")

        .join(
            broadcast(dim_date),
            to_date(col("r.review_creation_date")) == col("dd.date"),
            "inner"
        )

        .join(order_items, col("r.order_id") == col("oi.order_id"), "left")

        .join(
            broadcast(dim_product),
            col("oi.product_id") == col("dp.product_id"),
            "left"
        )

        .select(
            col("dd.date_sk"),
            col("dc.customer_sk"),

            coalesce(col("dp.product_sk"), lit(-1)).alias("product_sk"),

            col("r.review_id"),
            col("r.order_id"),

            # Dates
            to_date("r.review_creation_date").alias("review_creation_date"),
            to_date("r.review_answer_timestamp").alias("review_answer_timestamp"),

            # Measures
            col("r.review_score"),

            # Derived
            length(col("r.review_comment_message")).alias("review_length"),

            # Text
            col("r.review_comment_title"),
            col("r.review_comment_message")
        )
    )

    fact_reviews.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.gold.fact_reviews")

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "fact_reviews",
        "SUCCESS",
        start_time,
        end_time,
        0,
        "fact_reviews loaded successfully"
    )

except Exception as e:

    end_time = datetime.now()

    log_pipeline(
        "gold",
        "fact_reviews",
        "FAILED",
        start_time,
        end_time,
        0,
        str(e)
    )

    raise